# NAPE AICE 2026 Hackathon — Track 2: Intelligent Well Archive
## Verifiable Drilling & Wellbore Question-Answering Pipeline

### Workflow Purpose
This notebook provides a transparent, modular, and fully reproducible pipeline to answer technical questions from historical Volve Field drilling records.

### Step 1: Environment Setup & Dataset Verification
In this first step, we locate the preloaded dataset folders, verify that all daily drilling reports and wellbore documents are accessible, and confirm our working environment.

In [1]:
# Step 1: Locate the Dataset and Verify Document Folders
import os
from pathlib import Path

def locate_dataset_folder():
    """
    Finds the Track 2 data directory whether running from the repository
    root or from inside the submission folder.
    """
    possible_locations = [
        Path.cwd() / "data" / "track_2",
        Path.cwd().parent / "data" / "track_2",
        Path("data/track_2"),
        Path("../data/track_2")
    ]
    for folder in possible_locations:
        if folder.is_dir():
            return folder.resolve()
    raise FileNotFoundError("Could not locate 'data/track_2'. Please check dataset path.")

# 1. Define human-friendly folder paths
dataset_folder = locate_dataset_folder()
daily_reports_folder = dataset_folder / "documents" / "daily_drilling_reports" / "Daily_Drilling_Report___XML_Version"
wellbore_folder = dataset_folder / "documents" / "wellbore_files"
questions_file = dataset_folder / "open_test" / "open_test_questions.csv"

# 2. Inspect document counts for fast feedback
daily_report_files = sorted(daily_reports_folder.glob("*.xml"))
wellbore_files = [f for f in wellbore_folder.rglob("*") if f.is_file()]

# 3. Print clear feedback status
print("[OK] Dataset Environment Verified Successfully:")
print(f"     * Root Data Folder: {dataset_folder.name}")
print(f"     * Daily Drilling Reports (XML): {len(daily_report_files)} files found")
print(f"     * Wellbore Archive: {len(wellbore_files)} files found")
print(f"     * Questions File: {'Ready (' + questions_file.name + ')' if questions_file.is_file() else 'Missing'}")


[OK] Dataset Environment Verified Successfully:
     * Root Data Folder: track_2
     * Daily Drilling Reports (XML): 32 files found
     * Wellbore Archive: 180 files found
     * Questions File: Ready (open_test_questions.csv)


### Step 2: The Daily Drilling Report (XML) Reader
Here we define `read_daily_report(file_path)`, which parses a single WITSML XML file directly into memory.

It exhaustively extracts all drilling metrics:
- **Depths:** Current depth, TVD, kick-off depth, hole diameter, water depth
- **Fluids & Mud:** Sample time, location, mud type, density
- **Surveys:** Station depths, inclination angles, azimuth
- **Activities:** Operational timeline and comments (24h summary and forecast)
- **Casings & Pore Pressures:** Casing IDs/ODs and equivalent mud weight
- **Equipment & Logs:** Equipment failure logs and wireline service records
- **Geology & Bits:** Drill bit records, stratigraphy formations, and lithology shows

We test it immediately on a sample report (`15_9_19_A_1997_08_29.xml`) and pretty-print the extracted data as clean, indented JSON.

In [2]:
# Step 2: Clean & Complete Daily Drilling Report Reader (In-Memory Only)
import json
import xml.etree.ElementTree as ET

# Official WITSML XML namespace
xml_namespace = {"w": "http://www.witsml.org/schemas/1series"}

def get_clean_text(parent_element, tag_name):
    """
    Finds a tag and returns its text. Returns an empty string
    if the tag is missing or marked as '-999.99' (unrecorded in drilling standards).
    """
    if parent_element is None:
        return ""
    child_node = parent_element.find(tag_name, xml_namespace)
    if child_node is not None and child_node.text:
        text_value = child_node.text.strip()
        if text_value == "-999.99":
            return ""
        return text_value
    return ""

def read_daily_report(file_path):
    """
    Reads a single WITSML XML daily drilling report into a clean in-memory dictionary.
    Exhaustively extracts all operational metrics, fluids, surveys, activities, and logs.
    """
    xml_path = Path(file_path)
    root_node = ET.parse(xml_path).getroot()
    
    status_info = root_node.find(".//w:statusInfo", xml_namespace)
    wellbore_info = root_node.find(".//w:wellboreInfo", xml_namespace)
    
    # 1. Report Identity
    well_name = get_clean_text(root_node, ".//w:nameWell")
    report_date = get_clean_text(status_info, "w:dTim")[:10]  # Format: YYYY-MM-DD
    report_number = get_clean_text(status_info, "w:reportNo") or "1"
    rig_name = get_clean_text(wellbore_info, ".//w:rigAlias/w:name")
    spud_date = get_clean_text(wellbore_info, "w:dTimSpud")[:10]
    
    # 2. Key Operational Depths & Parameters
    current_depth = get_clean_text(status_info, "w:md")
    tvd_depth = get_clean_text(status_info, "w:tvd")
    kickoff_depth = get_clean_text(status_info, "w:mdKickoff")
    hole_diameter = get_clean_text(status_info, "w:diaHole")
    water_depth = get_clean_text(status_info, "w:waterDepth")
    summary_24h = get_clean_text(status_info, "w:sum24Hr")
    forecast_24h = get_clean_text(status_info, "w:forecast24Hr")
    
    # 3. Drilling Mud and Fluids Checks
    fluid_checks = []
    for fluid_tag in root_node.findall(".//w:fluid", xml_namespace):
        fluid_checks.append({
            "fluid_type": get_clean_text(fluid_tag, "w:type"),
            "sample_location": get_clean_text(fluid_tag, "w:locationSample"),
            "sample_time": get_clean_text(fluid_tag, "w:dTim")[11:16],  # HH:MM
            "density": get_clean_text(fluid_tag, "w:density")
        })
        
    # 4. Directional Survey Stations (Inclination and Depths)
    survey_stations = []
    for survey_tag in root_node.findall(".//w:surveyStation", xml_namespace):
        survey_stations.append({
            "depth_md": get_clean_text(survey_tag, "w:md"),
            "depth_tvd": get_clean_text(survey_tag, "w:tvd"),
            "inclination": get_clean_text(survey_tag, "w:incl"),
            "azimuth": get_clean_text(survey_tag, "w:azi")
        })
        
    # 5. Timeline Activities (24 Hours)
    activities = []
    for activity_tag in root_node.findall(".//w:activity", xml_namespace):
        activities.append({
            "start_time": get_clean_text(activity_tag, "w:dTimStart")[11:16],
            "end_time": get_clean_text(activity_tag, "w:dTimEnd")[11:16],
            "depth": get_clean_text(activity_tag, "w:md"),
            "comment": get_clean_text(activity_tag, "w:comments")
        })
        
    # 6. Pore Pressures
    pore_pressures = []
    for pp_tag in root_node.findall(".//w:porePressure", xml_namespace):
        pore_pressures.append({
            "depth_md": get_clean_text(pp_tag, "w:md"),
            "equivalent_mud_weight": get_clean_text(pp_tag, "w:equivalentMudWeight")
        })
        
    # 7. Casing and Tubing Records
    casing_records = []
    for casing_tag in root_node.findall(".//w:casing_liner_tubing", xml_namespace):
        casing_records.append({
            "outer_diameter": get_clean_text(casing_tag, "w:od"),
            "inner_diameter": get_clean_text(casing_tag, "w:id"),
            "grade": get_clean_text(casing_tag, "w:grade")
        })
        
    # 8. Equipment Failures (if logged)
    equipment_failures = []
    for eq_tag in root_node.findall(".//w:equipFailureInfo", xml_namespace):
        equipment_failures.append({
            "depth": get_clean_text(eq_tag, "w:md"),
            "description": get_clean_text(eq_tag, "w:description"),
            "downtime_hours": get_clean_text(eq_tag, "w:eTimMissProduction")
        })
        
    # 9. Wireline Logging Runs (if logged)
    wireline_logs = []
    for log_tag in root_node.findall(".//w:logInfo", xml_namespace):
        wireline_logs.append({
            "service_company": get_clean_text(log_tag, "w:serviceCompany"),
            "tool": get_clean_text(log_tag, "w:tool"),
            "run_number": get_clean_text(log_tag, "w:runNumber")
        })
        
    # 10. Gas Readings (if logged)
    gas_readings = []
    for gas_tag in root_node.findall(".//w:gasReadingInfo", xml_namespace):
        gas_readings.append({
            "reading_type": get_clean_text(gas_tag, "w:readingType"),
            "depth": get_clean_text(gas_tag, "w:mdTop"),
            "gas_high": get_clean_text(gas_tag, "w:gasHigh")
        })
        
    # 11. Drill Bit Records (if logged)
    bit_records = []
    for bit_tag in root_node.findall(".//w:bitRecord", xml_namespace):
        bit_records.append({
            "bit_number": get_clean_text(bit_tag, "w:numBit"),
            "size_in": get_clean_text(bit_tag, "w:diaBit"),
            "manufacturer": get_clean_text(bit_tag, "w:manufacturer"),
            "iadc_code": get_clean_text(bit_tag, "w:codeIADC"),
            "comment": get_clean_text(bit_tag, "w:comment")
        })
        
    # 12. Geological Stratigraphy / Formations (if logged)
    stratigraphy = []
    for strat_tag in root_node.findall(".//w:stratInfo", xml_namespace):
        stratigraphy.append({
            "formation_name": get_clean_text(strat_tag, "w:name"),
            "depth_md": get_clean_text(strat_tag, "w:mdTop")
        })
        
    # 13. Lithology Shows (if logged)
    lithology_shows = []
    for lith_tag in root_node.findall(".//w:lithShowInfo", xml_namespace):
        lithology_shows.append({
            "depth_md": get_clean_text(lith_tag, "w:mdTop"),
            "lithology": get_clean_text(lith_tag, "w:lithology")
        })
        
    return {
        "file_name": xml_path.name,
        "well_name": well_name,
        "report_date": report_date,
        "report_number": report_number,
        "rig_name": rig_name,
        "spud_date": spud_date,
        "current_depth_m": current_depth,
        "tvd_depth_m": tvd_depth,
        "kickoff_depth_m": kickoff_depth,
        "hole_diameter_in": hole_diameter,
        "water_depth_m": water_depth,
        "summary_24h": summary_24h,
        "forecast_24h": forecast_24h,
        "fluid_checks": fluid_checks,
        "survey_stations": survey_stations,
        "activities": activities,
        "pore_pressures": pore_pressures,
        "casing_records": casing_records,
        "equipment_failures": equipment_failures,
        "wireline_logs": wireline_logs,
        "gas_readings": gas_readings,
        "bit_records": bit_records,
        "stratigraphy": stratigraphy,
        "lithology_shows": lithology_shows
    }

# -------------------------------------------------------------------
# Fast Feedback Check: Pretty-print sample report as clean, indented JSON
# -------------------------------------------------------------------
sample_file = daily_reports_folder / "15_9_19_A_1997_08_29.xml"
sample_report = read_daily_report(sample_file)

print("[OK] Sample Daily Drilling Report (Parsed In-Memory):")
print(json.dumps(sample_report, indent=2))


[OK] Sample Daily Drilling Report (Parsed In-Memory):
{
  "file_name": "15_9_19_A_1997_08_29.xml",
  "well_name": "NO 15/9-19 A",
  "report_date": "1997-08-29",
  "report_number": "36",
  "rig_name": "BYFORD DOLPHIN",
  "spud_date": "1997-07-25",
  "current_depth_m": "4131",
  "tvd_depth_m": "3319",
  "kickoff_depth_m": "2178",
  "hole_diameter_in": "8.5",
  "water_depth_m": "84",
  "summary_24h": "OPERATIONS SUSPENDED DUE TO INDUSTRIAL ACTION. CONTINUED RIH WITH\nLOGGING TOOLS ON DRILL PIPE. PIPE STUCK WITH LOGGING TOOLS AT 4123M.\nWORKED PIPE. POOH WITH WIRELINE.",
  "forecast_24h": "WORK STUCK PIPE. POOH. R/U AND RUN 7\" CASING.",
  "fluid_checks": [
    {
      "fluid_type": "ULTIDRILL",
      "sample_location": "Active pit",
      "sample_time": "23:59",
      "density": "1.55"
    }
  ],
  "survey_stations": [],
  "activities": [
    {
      "start_time": "00:00",
      "end_time": "06:00",
      "depth": "2168",
      "comment": "OPERATIONS SUSPENDED DUE TO INDUSTRIAL ACTION. CO

### Step 3: Semantic Section Chunking Across the Archive
In this step, we loop through all 32 daily drilling reports and break them into clean, self-contained semantic chunks based on their WITSML operational sections (`statusInfo`, `activity`, `fluid`, `surveyStation`, `porePressure`, `casing_liner_tubing`, `equipFailureInfo`, `bitRecord`, `logInfo`).

Each chunk preserves:
- **`file_name`**: The source XML report
- **`well_name`**: Canonical well identity
- **`report_date`**: Exact report timestamp
- **`section_name`**: The specific section holding the fact
- **`evidence`**: Formatted citation string (`filename.xml, Section: <section>`)
- **`content`**: Human-readable text containing the facts and measurements

This creates our in-memory knowledge store ready for fast, exact retrieval.

In [3]:
# Step 3: Parse All Reports into Semantic Section Chunks (In-Memory Only)
from collections import Counter

all_report_chunks = []
section_counter = Counter()

for report_file in daily_report_files:
    report_data = read_daily_report(report_file)
    
    file_name = report_data["file_name"]
    well_name = report_data["well_name"]
    report_date = report_data["report_date"]
    
    # 1. Daily Status & Depths Chunk (Present in every report)
    status_content = (
        f"Well: {well_name} | Date: {report_date} | Rig: {report_data['rig_name']}\n"
        f"Current Depth: {report_data['current_depth_m']} mMD | TVD: {report_data['tvd_depth_m']} m | "
        f"Kick-Off Depth: {report_data['kickoff_depth_m']} mMD | Hole Diameter: {report_data['hole_diameter_in']} in\n"
        f"24h Summary: {report_data['summary_24h']}\n"
        f"24h Forecast: {report_data['forecast_24h']}"
    )
    all_report_chunks.append({
        "file_name": file_name,
        "well_name": well_name,
        "report_date": report_date,
        "section_name": "statusInfo",
        "evidence": f"{file_name}, Section: statusInfo",
        "content": status_content
    })
    section_counter["statusInfo"] += 1
    
    # 2. Timeline Activities Chunk
    if report_data["activities"]:
        activity_lines = [
            f"{act['start_time']}-{act['end_time']} (depth {act['depth']} m): {act['comment']}"
            for act in report_data["activities"]
        ]
        activity_content = f"Well: {well_name} | Date: {report_date} | Activities:\n" + "\n".join(activity_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "activity",
            "evidence": f"{file_name}, Section: activity",
            "content": activity_content
        })
        section_counter["activity"] += 1
        
    # 3. Drilling Mud and Fluids Chunk
    if report_data["fluid_checks"]:
        fluid_lines = [
            f"Sample at {fl['sample_time']} ({fl['sample_location']}): Type {fl['fluid_type']}, Density {fl['density']} g/cm3"
            for fl in report_data["fluid_checks"]
        ]
        fluid_content = f"Well: {well_name} | Date: {report_date} | Fluids:\n" + "\n".join(fluid_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "fluid",
            "evidence": f"{file_name}, Section: fluid",
            "content": fluid_content
        })
        section_counter["fluid"] += 1
        
    # 4. Directional Survey Stations Chunk
    if report_data["survey_stations"]:
        survey_lines = [
            f"Depth {s['depth_md']} mMD (TVD {s['depth_tvd']} m): Inclination {s['inclination']} deg, Azimuth {s['azimuth']} deg"
            for s in report_data["survey_stations"]
        ]
        survey_content = f"Well: {well_name} | Date: {report_date} | Surveys:\n" + "\n".join(survey_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "surveyStation",
            "evidence": f"{file_name}, Section: surveyStation",
            "content": survey_content
        })
        section_counter["surveyStation"] += 1
        
    # 5. Pore Pressures Chunk
    if report_data["pore_pressures"]:
        pp_lines = [
            f"Depth {pp['depth_md']} mMD: Equivalent Mud Weight (EMW) {pp['equivalent_mud_weight']} g/cm3"
            for pp in report_data["pore_pressures"]
        ]
        pp_content = f"Well: {well_name} | Date: {report_date} | Pore Pressure:\n" + "\n".join(pp_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "porePressure",
            "evidence": f"{file_name}, Section: porePressure",
            "content": pp_content
        })
        section_counter["porePressure"] += 1
        
    # 6. Casing & Tubing Records Chunk
    if report_data["casing_records"]:
        casing_lines = [
            f"OD: {csg['outer_diameter']} in, ID: {csg['inner_diameter']} in, Grade: {csg['grade']}"
            for csg in report_data["casing_records"]
        ]
        casing_content = f"Well: {well_name} | Date: {report_date} | Casing:\n" + "\n".join(casing_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "casing_liner_tubing",
            "evidence": f"{file_name}, Section: casing_liner_tubing",
            "content": casing_content
        })
        section_counter["casing_liner_tubing"] += 1
        
    # 7. Equipment Failures Chunk (if present)
    if report_data["equipment_failures"]:
        eq_lines = [
            f"Depth {eq['depth']} m: {eq['description']} (Downtime: {eq['downtime_hours']} hrs)"
            for eq in report_data["equipment_failures"]
        ]
        eq_content = f"Well: {well_name} | Date: {report_date} | Equipment Failures:\n" + "\n".join(eq_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "equipFailureInfo",
            "evidence": f"{file_name}, Section: equipFailureInfo",
            "content": eq_content
        })
        section_counter["equipFailureInfo"] += 1
        
    # 8. Drill Bit Chunk (if present)
    if report_data["bit_records"]:
        bit_lines = [
            f"Bit #{b['bit_number']} ({b['size_in']} in, {b['manufacturer']}): {b['comment']}"
            for b in report_data["bit_records"]
        ]
        bit_content = f"Well: {well_name} | Date: {report_date} | Bit Record:\n" + "\n".join(bit_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "bitRecord",
            "evidence": f"{file_name}, Section: bitRecord",
            "content": bit_content
        })
        section_counter["bitRecord"] += 1
        
    # 9. Wireline Logs Chunk (if present)
    if report_data["wireline_logs"]:
        log_lines = [
            f"Run #{l['run_number']}: {l['service_company']} (Tool: {l['tool']})"
            for l in report_data["wireline_logs"]
        ]
        log_content = f"Well: {well_name} | Date: {report_date} | Wireline Logs:\n" + "\n".join(log_lines)
        all_report_chunks.append({
            "file_name": file_name,
            "well_name": well_name,
            "report_date": report_date,
            "section_name": "logInfo",
            "evidence": f"{file_name}, Section: logInfo",
            "content": log_content
        })
        section_counter["logInfo"] += 1

# -------------------------------------------------------------------
# Fast Feedback Check: Print chunk counts and a pretty JSON preview
# -------------------------------------------------------------------
print(f"[OK] Successfully Generated {len(all_report_chunks)} Semantic Chunks Across All 32 Reports:")
for section_name, count in section_counter.most_common():
    print(f"     * Section '{section_name}': {count} chunks")

# Sample chunk preview (a fluid chunk)
sample_chunk = [c for c in all_report_chunks if c["section_name"] == "fluid"][0]
print("\n--- Sample Chunk Preview (JSON Format) ---:")
print(json.dumps(sample_chunk, indent=2))


[OK] Successfully Generated 106 Semantic Chunks Across All 32 Reports:
     * Section 'statusInfo': 32 chunks
     * Section 'activity': 31 chunks
     * Section 'fluid': 21 chunks
     * Section 'surveyStation': 7 chunks
     * Section 'porePressure': 5 chunks
     * Section 'equipFailureInfo': 4 chunks
     * Section 'casing_liner_tubing': 2 chunks
     * Section 'bitRecord': 2 chunks
     * Section 'logInfo': 2 chunks

--- Sample Chunk Preview (JSON Format) ---:
{
  "file_name": "15_9_19_A_1997_08_29.xml",
  "well_name": "NO 15/9-19 A",
  "report_date": "1997-08-29",
  "section_name": "fluid",
  "evidence": "15_9_19_A_1997_08_29.xml, Section: fluid",
  "content": "Well: NO 15/9-19 A | Date: 1997-08-29 | Fluids:\nSample at 23:59 (Active pit): Type ULTIDRILL, Density 1.55 g/cm3"
}
